# Famílias avançadas e ajuste de hiperparâmetros

Além do híbrido residual `ŷ = b + α·r̂`, a busca pode testar três estruturas:

| Família | Estratégia | Ideia |
|---|---|---|
| `residual` | `residual_multistage` | correção da correção: cada estágio aprende o erro do modelo anterior (até 3) |
| `decomposition` | `decomposition` | `y = suave + volátil`; a base prevê a parte suave e um corretor a volátil. `ma`: média móvel **causal** com janela pela curtose; `stl`: só na variante A |
| `combination` | `fixed_combination` | média ponderada de duas bases; pesos iguais ou pelo inverso do erro fora da amostra no treino |

Todas obedecem às mesmas regras de causalidade: nada depois da origem entra no ajuste, nas janelas ou nos pesos. A seleção incremental trata cada uma como mais uma família: só entra se ganhar do campeão atual pela regra do protocolo.

No fim, `tune_model` ajusta os hiperparâmetros de uma base **só em `V_search`**.

In [ ]:
import autohybts

print("AutoHybTS", autohybts.__version__)

## 1. Experimento

In [ ]:
from autohybts import AutoHybridExperiment
from autohybts.datasets import get_data

y = get_data("synthetic_daily", n=500)
exp = AutoHybridExperiment(
    fh=14,
    session_id=42,
    fold=3,
    base_models=["naive", "seasonal_naive", "linear", "knn"],
    correctors=["linear"],
    residual_windows=[7],
    alphas=[1.0],
).fit(y)

## 2. Montar cada estrutura à mão

Cada chamada avalia a estrutura em todas as origens (`V_search` e `V_select`).

In [ ]:
staged = exp.create_hybrid("seasonal_naive", "linear", extra_stages=["linear"])
decomposition = exp.create_decomposition("linear", "linear", method="ma")
combination = exp.create_combination(["seasonal_naive", "linear"], weights="inverse_mse")

for result in (staged, decomposition, combination):
    print(f"{result.metrics['MASE']:.3f}  {result.candidate.label}")

Os diagnósticos de cada origem mostram o que foi aprendido só com o treino: a janela da média móvel escolhida pela curtose e os pesos da combinação.

In [ ]:
import pandas as pd

pd.DataFrame(
    {
        "ma_window": [t.diagnostics.get("ma_window") for t in decomposition.evaluation.trials],
        "peso_seasonal_naive": [
            t.diagnostics.get("weight_0") for t in combination.evaluation.trials
        ],
        "peso_linear": [t.diagnostics.get("weight_1") for t in combination.evaluation.trials],
    },
    index=[t.fold_id for t in decomposition.evaluation.trials],
)

## 3. Busca com todas as famílias

`include` escolhe as famílias. Estágios extras entram no refinamento S3; decomposições e combinações numa etapa S4, com o orçamento que sobrar (ao menos uma candidata por família).

In [ ]:
result = exp.compare_models(
    include=("standalone", "residual", "multistage", "decomposition", "combination")
)
result.best

In [ ]:
ledger = result.search.ledger_frame()
ledger.groupby(["stage", "strategy", "status"]).size()

Decisões da cadeia incremental, família por família:

In [ ]:
result.decisions[["family", "champion_before", "challenger", "accepted", "median_gain", "win_rate"]]

## 4. Comparar as estratégias de montar o sistema

Cada candidata avaliada na busca vira um ponto: erro em `V_search` contra custo (ou complexidade, o número de modelos ajustados). A **fronteira de Pareto** liga as candidatas que nenhuma outra supera nos dois eixos: mostra o que cada passo de complexidade comprou. É relatório; a escolha do campeão continua sendo a regra do protocolo em `V_select`.

In [ ]:
result.strategies

In [ ]:
result.plot("pareto")

In [ ]:
result.plot("pareto", x="complexity")

In [ ]:
result.plot("strategies")

In [ ]:
table = result.strategy_table()
table[table["pareto_cost"]][["name", "strategy", "score", "cost_seconds", "complexity"]]

## 5. Ajuste de hiperparâmetros

`tune_model` sorteia configurações da base (Optuna/TPE se o extra `optuna` estiver instalado; senão busca aleatória com semente) e pontua cada uma só em `V_search`. A configuração original é a tentativa 0, então o resultado nunca é pior que ela nessa camada. O `V_select` do modelo ajustado não foi visto pelo ajuste: compare nele.

In [ ]:
tuned, tuner = exp.tune_model("knn", n_trials=15, return_tuner=True)
print(
    tuner.sampler,
    "| melhor:",
    tuner.best.label,
    "| ganho em V_search:",
    round(tuner.improvement, 3),
)
tuner.trials.sort_values("score").head()

In [ ]:
original = exp.create_model("knn", lags=7)
pd.DataFrame(
    {
        "original": original.evaluation.to_frame().groupby("layer")["MASE"].mean(),
        "ajustado": tuned.evaluation.to_frame().groupby("layer")["MASE"].mean(),
    }
)

Espaço próprio: `(mínimo, máximo)` para intervalos, lista para opções, nomes com ponto para parâmetros do estimador.

In [ ]:
exp.tune_model(
    "svm",
    n_trials=6,
    search_space={
        "lags": (7, 21),
        "estimator_params.C": (0.1, 10.0),
        "estimator_params.epsilon": [0.01, 0.1],
    },
)